# KhaledDSML 17 — Leak-safe ML Evaluation
## التحقق المتداخل ومراقبة جودة تقييم نماذج تعلم الآلة

**Research teaching edition:** `0.10.0.dev0` (GitHub branch), not published on PyPI. / **نسخة تطويرية:** لا تزال 0.9.0 هي النسخة المنشورة.

**All records synthetic.** Independent rows are assumed; never substitute random CV for time/group-aware validation.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@feature/v0.10-ml-validation-expansion"

## 1 — Synthetic dataset and preprocessing Pipeline / بيانات تجريبية ومنع تسرب المعالجة

التقييس يجري داخل Pipeline وليس على كامل الجدول قبل التقسيم. / Scaling is fitted only in training folds.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from khaled_dsml import (
    nested_cv_model_evaluation, permutation_score_significance,
    holdout_permutation_importance, learning_curve_diagnostic,
    dummy_baseline_cv, display_analysis_dashboard,
)
features, target = make_classification(
    n_samples=180, n_features=5, n_informative=3, n_redundant=0,
    random_state=13,
)
X = pd.DataFrame(features, columns=["economy","health","education","market","region"])
X_train, X_test, y_train, y_test = train_test_split(
    X, target, test_size=.25, stratify=target, random_state=15
)
def build_pipeline():
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=400, solver="liblinear")),
    ])

## 2 — Nested CV / التحقق المتداخل

المعاملات تُختار داخل التدريب، ثم تقاس على طيات خارجية منفصلة. لا ينشأ أي نموذج إنتاج نهائي تلقائيًا.

In [ ]:
nested = nested_cv_model_evaluation(
    build_pipeline(), X_train, y_train, {"model__C":[.1,1.,10.]},
    outer_cv=3,inner_cv=3,scoring="accuracy",random_state=31
)
display_analysis_dashboard(nested,theme="amethyst")

## 3 — Label permutation score / اختبار تبديل ملصقات Y

قد تحتاج زيادة عدد التبديلات إلى وقت أطول. قيمة p تعتمد على قابلية تبادل الملصقات تحت H0. / Permutation inference requires exchangeability and completely refitted CV pipelines.

In [ ]:
permuted = permutation_score_significance(
    build_pipeline(), X_train, y_train, scoring="accuracy",
    cv=3, n_permutations=39,random_state=31
)
display_analysis_dashboard(permuted,theme="ocean")

## 4 — Independent holdout importance / أهمية السمات على مجموعة اختبار مستقلة

لا يُستخدم `X_test` في التدريب أو اختيار المعلمات في هذا المثال؛ أهمية التبديل ليست تأثيرًا سببيًا. / A holdout importance score is not causal.

In [ ]:
fitted = build_pipeline().fit(X_train, y_train)
importance = holdout_permutation_importance(
    fitted, X_test, y_test, scoring="accuracy",
    n_repeats=10,random_state=14
)
display_analysis_dashboard(importance,theme="aurora")

## 5 — Learning curves / منحنيات التعلّم

تصف الفرق بين أداء التدريب والتحقق داخل CV؛ ليست فاصل ثقة أو تقييمًا خارجيًا مستقلًا.

In [ ]:
curve = learning_curve_diagnostic(
    build_pipeline(), X_train, y_train,
    cv=3,scoring="accuracy",train_sizes=(.5,.75,1.),random_state=25
)
display_analysis_dashboard(curve,theme="amethyst")

## 6 — Dummy model baseline / مقارنة بخط الأساس الساذج

تُحسب الأكثرية داخل طيات التدريب فقط، ولا تستخدم الملصقات الخارجية لضبط خط الأساس.

In [ ]:
baseline = dummy_baseline_cv(
    X_train,y_train,task="classification",
    scoring="accuracy",cv=3,random_state=25
)
display_analysis_dashboard(baseline,theme="ocean")

## Scientific caution / محاذير التفسير

- Do not standardize/encode/select features globally before CV.
- Use GroupKFold/TimeSeriesSplit alternatives when subjects or timestamps are dependent; these functions currently target iid samples.
- The permutation p-value is conditional on label exchangeability and is not corrected for selective hypothesis searches.
- Feature importance is descriptive; highly correlated features can split the effect and perturbation changes the input distribution.

**للبحوث المنشورة:** راجع التصميم، استقلال العينات، صيغ المقاييس، ومصدر البيانات قبل اعتماد النتائج.